# Bi-level pruning - CIFAR-10 (VGG16-BN / ResNet-56)

Doi chieu voi so **published** cua L1 / HRank / GAL / SSS / CORING / SPSRC ma khong
phai chay lai tung baseline. Protocol lay tu repo CORING (github.com/vantienpham/CORING).

| Truc fair | Cach xu ly |
|---|---|
| Transform | RandomCrop(32,pad=4) + HFlip, std **(.2023,.1994,.2010)** - dung `main/data/cifar10.py` |
| Kien truc | ResNet-56 shortcut **option A** (zero-pad) = 0.853M, dung nhu HRank |
| Diem xuat phat | Nap thang `resnet_56.pt` cua HRank - **cung checkpoint voi CORING** |
| Recipe finetune | `coring`: 300 ep, lr 0.01, x0.1@150,225, wd 5e-3, bs 128 |
| Ngan sach epoch | Vong prune tieu bao nhieu thi finetune tru bay nhieu -> tong = 300 |
| Muc nen | `--match-macs` tu tim target-sparsity de rot dung diem cua CORING |
| Loai method | Chay ca `structured-only` (cung loai CORING) va `bi-level` (ablation) |

**Thu tu:** Cell 1-4 setup -> Cell 5 SMOKE -> Cell 6 dense (bo qua neu co pretrained) -> Cell 7 chay that.

**Truoc khi chay:** Settings > Accelerator > **GPU**, Settings > **Internet ON**.


In [ ]:
# --- 1. Repo + branch dev ---
import os, subprocess, sys, json, time

REPO = '/kaggle/working/OnestageDetectionPunner'
BRANCH = 'dev'

if not os.path.isdir(REPO):
    # repo private -> dung token, hoac Add Data roi copy vao /kaggle/working
    # subprocess.run(f'git clone https://<TOKEN>@github.com/barone04/OnestageDetectionPunner.git {REPO}',
    #                shell=True, check=True)
    raise SystemExit(f'{REPO} chua co. Clone repo hoac Add Data truoc.')

os.chdir(REPO)
subprocess.run(f'git fetch --all -q && git checkout {BRANCH} -q && git pull -q', shell=True)
subprocess.run('git log --oneline -1', shell=True)
print('cwd:', os.getcwd())


In [ ]:
# --- 2. .env -> os.environ (WANDB_API_KEY). Bo qua neu khong dung wandb ---
ENV = '/kaggle/input/datasets/bophaninhthi/env-file1/.env'
USE_WANDB = os.path.isfile(ENV)
if USE_WANDB:
    for line in open(ENV):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip().strip(chr(34)).strip(chr(39))
    USE_WANDB = bool(os.environ.get('WANDB_API_KEY'))
os.environ.setdefault('WANDB_PROJECT', 'bilevel_cifar10')
print('wandb:', 'ON' if USE_WANDB else 'OFF')


In [ ]:
# --- 3. Deps + kiem tra moi truong ---
subprocess.run('pip -q install thop' + (' wandb' if USE_WANDB else ''), shell=True)
import torch, torchvision, thop
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Bat GPU: Settings > Accelerator > GPU'


In [ ]:
# --- 4. Config ---
MODEL    = 'resnet56'    # 'resnet56' hoac 'vgg16'
VGG_HEAD = 'hrank'       # chi cho vgg16. hrank=14.99M (HRank/CORING) | single=14.72M (SPSRC)
PROTOCOL = 'coring'      # recipe finetune: 'coring' (300ep) | 'spsrc' (80ep)
SEED     = 0

# Checkpoint pretrained cua HRank -> CUNG diem xuat phat voi CORING.
# Upload resnet_56.pt len Kaggle roi tro vao day. None -> tu train dense (Cell 6).
PRETRAINED = '/kaggle/input/hrank-ckpt/resnet_56.pt'

# Diem nen can match, lay tu BANG PUBLISHED cua CORING (don vi: MACs trieu).
# Doi sang params bang MATCH_METRIC='params' neu bang cua ho bao params.
TARGETS = [62.7]          # vd 1 diem/session; them khi con quota
MATCH_METRIC = 'macs'     # 'macs' | 'params'

# Hai bien the: structured-only dat canh CORING trong bang chinh,
# bi-level day du de o bang ablation.
VARIANTS = {'structured-only': '--no-unstructured', 'bi-level': ''}

DATA = '/kaggle/working/data'
OUT  = '/kaggle/working/output/cifar'
DENSE_DIR = f'{OUT}/{MODEL}_dense'
os.makedirs(DATA, exist_ok=True)

WB = '--wandb' if USE_WANDB else ''
head = f'--vgg-head {VGG_HEAD}' if MODEL == 'vgg16' else ''

def run(cmd):
    """Chay lenh, in truc tiep, dung han neu that bai."""
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'That bai (exit {r.returncode}): {cmd}')

print(f'{MODEL} | protocol={PROTOCOL} | targets={TARGETS} {MATCH_METRIC}')
print('pretrained:', PRETRAINED or '(tu train dense)')


## 5. Smoke test

Kiem tra truoc khi dot GPU:
1. Model dung so params published (ResNet-56 A = 0.853M, VGG16-BN hrank = 14.99M)
2. Surgery cho lean model **giong het** masked model (`max|diff| < 1e-7`)
3. Checkpoint pretrained nap duoc, khong thieu/thua key
4. Ca 3 mode chay thong voi 1 epoch

~5 phut. Fail o day thi dung.


In [ ]:
# --- 5a. Self-check model + surgery ---
run('python -m models.cifar')
run('python -m pruning.surgery_cifar')


In [ ]:
# --- 5b. Checkpoint pretrained co nap duoc khong (raise neu lech key) ---
if PRETRAINED:
    assert os.path.isfile(PRETRAINED), PRETRAINED
    sys.path.insert(0, REPO)
    from models.cifar import build_cifar_model, load_hrank_state_dict
    import cifar as C
    _m = build_cifar_model(C.build(MODEL, 10, 'all', VGG_HEAD).config())
    load_hrank_state_dict(_m, PRETRAINED)
    print('params:', sum(p.numel() for p in _m.parameters())/1e6, 'M')
else:
    print('Khong co pretrained -> se train dense o Cell 6')


In [ ]:
# --- 5c. Smoke: dense 1 epoch (--skip-gate vi 1 epoch chac chan khong dat moc) ---
SMOKE = f'{OUT}/_smoke'
run(f'python cifar.py --mode dense --model {MODEL} {head} --data-path {DATA} '
    f'--epochs 1 --skip-gate --output-dir {SMOKE}_dense')


In [ ]:
# --- 5d. Smoke: prune (2 vong x 1 ep) + match ratio + surgery ---
src = f'--pretrained {PRETRAINED}' if PRETRAINED else f'--checkpoint {SMOKE}_dense/model_best.pth'
run(f'python cifar.py --mode prune --model {MODEL} {head} --data-path {DATA} '
    f'{src} --protocol {PROTOCOL} --match-{MATCH_METRIC} {TARGETS[0]} '
    f'--prune-iters 2 --prune-finetune-epochs 1 --output-dir {SMOKE}_p')
print(json.load(open(f'{SMOKE}_p/cost.json')))


In [ ]:
# --- 5e. Smoke: finetune 1 epoch tren lean model ---
run(f'python cifar.py --mode finetune --data-path {DATA} --protocol {PROTOCOL} '
    f'--lean {SMOKE}_p/model_lean.pth --epochs 1 --output-dir {SMOKE}_ft')
print(chr(10) + 'SMOKE TEST PASS - duoc phep chay that.')


## 6. Dense baseline (chi khi KHONG co pretrained)

Co `PRETRAINED` thi **bo qua cell nay** - dung checkpoint cua HRank la fair hon han,
vi CORING cung xuat phat tu dung file do chu khong tu train.

Neu van train: ResNet-56 200 ep ~1-1.5h. Cell tu **dung** neu top-1 lech >0.5% so voi
published (ResNet-56 93.26% | VGG16-BN 93.96%) - prune tren dense sai moc thi khong
duoc trich bang cua ho.


In [ ]:
# --- 6. Dense (bo qua neu da co pretrained) ---
if not PRETRAINED:
    t0 = time.time()
    run(f'python cifar.py --mode dense --model {MODEL} {head} --data-path {DATA} '
        f'--seed {SEED} --output-dir {DENSE_DIR} {WB}')
    print(f'Dense xong sau {(time.time()-t0)/3600:.2f}h')
    SRC = f'--checkpoint {DENSE_DIR}/model_best.pth'
else:
    SRC = f'--pretrained {PRETRAINED}'
print('nguon dense:', SRC)


## 7. Prune + finetune

**Match iso-FLOPs:** `--match-macs` tu do nhi phan tim `target-sparsity` sao cho lean
model rot dung diem nen cua CORING. Khong ton GPU - so kenh con lai chi phu thuoc
prune_ratio nen tinh duoc ma khong can train. In ra do lech %, canh bao neu >2%.

**Ngan sach epoch:** vong prune tieu `prune_iters x prune_finetune_epochs` = 15 ep,
finetune con 285 ep, milestone dich ve 135/210 -> tong 300 ep, **bang dung CORING**
(one-shot + 300 ep finetune).

**Hai bien the:**

| | Muc unstructured | Dat o dau |
|---|---|---|
| `structured-only` | tat | **bang chinh**, canh CORING - cung loai method |
| `bi-level` | bat | bang ablation - muc unstructured them duoc gi |

Ly do phai tach: surgery chi cat kenh, nen zero cua muc unstructured van nam trong
lean model va thop dem du. Bi-level tra gia accuracy cho muc do ma **khong duoc
cong gi vao params/MACs** - dat thang canh CORING la tu trói tay.

~1.5-2h moi (target x bien the). 1 target x 2 bien the ~ 3-4h. Canh quota 12h/session.


In [ ]:
# --- 7. Chay that: moi target x moi bien the ---
results = {}
for tgt in TARGETS:
    for vname, vflag in VARIANTS.items():
        tag = f'{MODEL}_{MATCH_METRIC}{tgt}_{vname}'
        p_dir, f_dir = f'{OUT}/{tag}', f'{OUT}/{tag}_ft'
        print(chr(10) + '=' * 64 + chr(10) + f' {tag}' + chr(10) + '=' * 64, flush=True)

        run(f'python cifar.py --mode prune --model {MODEL} {head} --data-path {DATA} '
            f'{SRC} --protocol {PROTOCOL} --seed {SEED} {vflag} '
            f'--match-{MATCH_METRIC} {tgt} '
            f'--prune-iters 5 --prune-finetune-epochs 3 --output-dir {p_dir}')

        run(f'python cifar.py --mode finetune --data-path {DATA} --protocol {PROTOCOL} '
            f'--seed {SEED} --lean {p_dir}/model_lean.pth --output-dir {f_dir} {WB}')

        cost = json.load(open(f'{p_dir}/cost.json'))
        ck = torch.load(f'{f_dir}/model_best.pth', map_location='cpu', weights_only=False)
        results[tag] = dict(cost, top1=ck['top1'])
        print(f"{tag} -> top1={ck['top1']:.2f}%  {cost}")


In [ ]:
# --- 8. Bang tong ket (dan thang vao .tex duoc) ---
cols = ('bien the', 'ratio', 'top1', 'params M', '-params%', 'MACs M', '-MACs%')
print('{:<16} | {:>5} | {:>7} | {:>8} | {:>8} | {:>8} | {:>7}'.format(*cols))
print('-' * 78)
for tag, r in results.items():
    print('{:<16} | {:>5} | {:>6.2f}% | {:>8.3f} | {:>7.2f}% | {:>8.2f} | {:>6.2f}%'.format(
        r['variant'], r['target_sparsity'], r['top1'], r['params_M'],
        r['params_red_pct'], r['macs_M'], r['macs_red_pct']))
path = f'{OUT}/summary_{MODEL}_{PROTOCOL}.json'
json.dump(results, open(path, 'w'), indent=2)
print(chr(10) + 'Luu: ' + path)
